In [241]:
import math

In [242]:
class Value:

    def __init__(self, data, prev = None):
        self.data = data
        self.prev = prev if prev is not None else ()
        self.grad = 0
        # why need this now 
        #we need to send our grad (calculate and send)
        #to our ancestors this function takes our grad and calulate their grad and accumulates
        #the grad with respect to loss through this node
        #when our descendants call backward they asssign us grad
        #we call grad to assign our ancestors grad (assign == accumulate)
        #_backward i know how to contribute to descendants grads using my grad
        self._backward = lambda : None

    def __repr__(self):
        return (f"Value(data={self.data},grad={self.grad})")
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, prev =(self, other))
        def backward():
            self.grad +=  out.grad
            other.grad += out.grad
        out._backward = backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, prev = (self, other))
        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = backward
        return out


    def __radd__(self, other):
        return self.__add__(other)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __neg__(self):
        return self.__mul__(-1.0)


    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, prev = (self,))
        def backward():
            self.grad += (other * (self.data ** (other -1))) * out.grad
        out._backward = backward
        return out

    def __sub__(self, other):
        return self.__add__(-other)


    def __truediv__(self, other):
        #other could be value type or int/float type
        #so don't directly call other__pow__()
        #instead use operator ** which invokes
        #whatever needed accordingly
        return self.__mul__(other ** -1)


    def __rtruediv__(self, other):
        #other might be a scalar (int, float) sclar/Value calls this so
        return Value(other) / self


    def __rsub__(self, other):
        return Value(other) - self

    def exp(self):
        out = Value(math.exp(self.data), prev = (self,))
        def backward():
            self.grad += out.data * out.grad 
        out._backward = backward
        return out


    def sigmoid(self):
        out = 1/(1+(self.exp() ** -1))
        return out

    def relu(self):
        out =  Value(self.data, prev = (self,)) if self.data > 0 else Value(0, prev = (self,))
        def backward():
            self.grad = (1 if self.data > 0 else 0) * out.grad
        out._backward = backward
        return out

# x -> e^x -> 1/1+y^-1

    def backward(self):
        nodes = []
        visited = set()
        def visit(node):
            if node not in visited:
                visited.add(node)
                for child in node.prev:
                        visit(child)
                nodes.append(node)

        visit(self)
        for node in nodes:
            node.grad = 0

        self.grad = 1
        for node in reversed(nodes):
            node._backward()
        print(nodes)

In [243]:
x = Value(2)
w = Value(3)
b = Value(5)
xw = x * w
y = xw + b
l = y - 12

In [244]:
l.backward()

[Value(data=2,grad=3), Value(data=3,grad=2), Value(data=6,grad=1), Value(data=5,grad=1), Value(data=11,grad=1), Value(data=-12,grad=1), Value(data=-1,grad=1)]


In [245]:
class Modlue:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0  

In [246]:
import random
class Neuron(Modlue):
    def __init__(self, nin, nonlin = True):
        self.weights = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.bias =  Value(0)
        self.nonlin = nonlin

    def __call__(self, inputs:list):
        out = sum((w*x for w,x in zip(self.weights, inputs)), self.bias)
        return out.relu() if self.nonlin else out

    def parameters(self):
        return [w for w in self.weights] + [self.bias]

In [247]:
class Layer(Modlue):
    def __init__(self, nin, neurons):
        self.neurons = [Neuron(nin) for _  in range(neurons)]

    def __call__(self, inputs:list):
        return [n(inputs) for n in self.neurons]

    def parameters(self):
        return [p  for n in self.neurons for p in n.parameters()]

class MLP(Modlue):
    def __init__(self, nin, layers:list): #layers contain number of neurons in each layer
        self.layers = [Layer(i, j) for i,j in zip([nin]+layers, layers)]

    def __call__(self, inputs):
        out = inputs
        for layer in self.layers:
            out = layer(out)
        return out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
       

In [248]:
mlp = MLP(3, [2, 1])
mlp([1, 2, 3])


[Value(data=1.445986098913696,grad=0)]

In [249]:
inputs = [-1, 1]
targets = [0, 1]
MLP(2, [3, 2, 1])
mlp([3,3])



[Value(data=0,grad=0)]

In [250]:
v = Value(-3).relu()
v.backward()

[Value(data=-3,grad=0), Value(data=0,grad=1)]
